<a href="https://colab.research.google.com/github/Huzaifasura/upscaler/blob/main/Copy_of_Bulk_Image_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Install CodeFormer and Download Model { display-mode: "form" }
from IPython.display import clear_output
%cd /content
!rm -rf /content/CodeFormer
!git clone https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -r /content/CodeFormer/requirements.txt
# Install basicsr
!python /content/CodeFormer/basicsr/setup.py develop
# Download the pre-trained model
!python /content/CodeFormer/scripts/download_pretrained_models.py facelib
!python /content/CodeFormer/scripts/download_pretrained_models.py CodeFormer
clear_output()

In [2]:
#@title Utils { display-mode: "form" }


# Visualization function
import cv2
import matplotlib.pyplot as plt
import os
import shutil
def display(img1, img2):
  fig = plt.figure(figsize=(25, 10))
  ax1 = fig.add_subplot(1, 2, 1)
  plt.title('Input', fontsize=16)
  ax1.axis('off')
  ax2 = fig.add_subplot(1, 2, 2)
  plt.title('CodeFormer', fontsize=16)
  ax2.axis('off')
  ax1.imshow(img1)
  ax2.imshow(img2)
def imread(img_path):
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  return img

upload_folder = '/content/CodeFormer/inputs/user_upload'
if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
os.mkdir(upload_folder)
if os.path.exists("/content/CodeFormer/results/user_upload_0.7"):
  shutil.rmtree("/content/CodeFormer/results/user_upload_0.7")
clear_output()

In [3]:
#@title Upload single image or bulk images  { display-mode: "form" }

from PIL import Image
import os
import shutil
# from google.colab import drive # Removed Google Drive import
import os
from google.colab import files
keep_images = False #@param {type:"boolean"}
upload_folder = '/content/CodeFormer/inputs/user_upload'
%cd /content/CodeFormer

if keep_images:
  pass
else:
  if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
  os.mkdir(upload_folder)


# Removed image_from_drive function

def image_from_device():
  uploaded = files.upload()
  for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'move {filename} to {dst_path}')
    shutil.move(filename, dst_path)

upload_from = "local device" #@param ["local device"] {allow-input: true} # Removed Google Drive option
# if upload_from == "google drive": # Removed Google Drive option
#   image_from_drive() # Removed Google Drive option
if upload_from == "local device":
  image_from_device()
clear_output()
file_count = len(os.listdir(upload_folder))
if file_count >= 1:
    print("Run next cell")
else:
    print("Please upload an image.")

Run next cell


In [6]:
#@title Sequential Processing and Instant Download { display-mode: "form" }

import os
import time
from google.colab import files

CODEFORMER_FIDELITY = 0.7 #@param {type:"slider", min:0, max:1, step:0.01}
BACKGROUND_ENHANCE = True #@param {type:"boolean"}
FACE_UPSAMPLE = True #@param {type:"boolean"}
UPSAMPLER = "4" #@param ["2", "4", "6"]

upscale_command = ""
if BACKGROUND_ENHANCE:
  if FACE_UPSAMPLE:
    upscale_command = "--bg_upsampler realesrgan --face_upsample"
  else:
    upscale_command = "--bg_upsampler realesrgan"
else:
  if FACE_UPSAMPLE:
    upscale_command = "--face_upsample"

if BACKGROUND_ENHANCE or FACE_UPSAMPLE:
  upscale_command += f" --upscale {UPSAMPLER}"

input_folder = '/content/CodeFormer/inputs/user_upload'
output_parent = f"/content/CodeFormer/results/user_upload_{CODEFORMER_FIDELITY}/final_results"

if os.path.exists(input_folder) and len(os.listdir(input_folder)) > 0:
    # Sort input images to maintain chronological/alphabetical sequence
    sorted_inputs = sorted(os.listdir(input_folder))
    image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp")
    sorted_inputs = [f for f in sorted_inputs if f.lower().endswith(image_extensions)]

    print(f"Found {len(sorted_inputs)} images. Processing and downloading them sequentially...\n")

    for i, filename in enumerate(sorted_inputs, 1):
        print(f"\n--- [{i}/{len(sorted_inputs)}] Processing: {filename} ---")
        single_input_path = os.path.join(input_folder, filename)

        # Process only this specific image
        cmd = f"python inference_codeformer.py -w {CODEFORMER_FIDELITY} --input_path \"{single_input_path}\" {upscale_command}"
        os.system(cmd)

        # Find the processed image and download it instantly
        processed_image_path = os.path.join(output_parent, filename)
        if os.path.exists(processed_image_path):
            print(f"Downloading processed image: {filename}")
            files.download(processed_image_path)
            time.sleep(1.0) # Small delay to avoid browser blocking multiple downloads
        else:
            # Some configurations might output with a slightly different name structure
            print(f"Could not find direct output for {filename}, checking output folder...")
            if os.path.exists(output_parent):
                output_files = os.listdir(output_parent)
                if output_files:
                    # Download newest file in output directory
                    latest_file = max([os.path.join(output_parent, f) for f in output_files], key=os.path.getctime)
                    print(f"Downloading latest output: {os.path.basename(latest_file)}")
                    files.download(latest_file)
                    time.sleep(1.0)
else:
    print("No input images found. Please upload images in the previous cell first.")

Found 1 images. Processing and downloading them sequentially...


--- [1/1] Processing: Advertising_Icons__A_clean_layout_of_exactly_10_fl_33_1.png ---
Could not find direct output for Advertising_Icons__A_clean_layout_of_exactly_10_fl_33_1.png, checking output folder...


In [7]:
import os
from google.colab import files

# Let's inspect the entire results directory to find where the output was generated
base_results_path = '/content/CodeFormer/results'
if os.path.exists(base_results_path):
    print("--- Files and directories inside results folder: ---")
    for root, dirs, files_list in os.walk(base_results_path):
        for file in files_list:
            full_path = os.path.join(root, file)
            print(full_path)
            # If it's an image file, let's try to download it directly
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                print(f"Downloading found result: {full_path}")
                files.download(full_path)
else:
    print("Results directory does not exist. The inference command might have failed. Let's list the inputs to double-check:")
    if os.path.exists('/content/CodeFormer/inputs/user_upload'):
        print(os.listdir('/content/CodeFormer/inputs/user_upload'))

--- Files and directories inside results folder: ---
/content/CodeFormer/results/test_img_0.7/final_results/Advertising_Icons__A_clean_layout_of_exactly_10_fl_33_1.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

it's your choice

 1. ```"zip"``` if you upload many images

 2. ```"single file"``` if you upload 1-2 images


In [5]:
#@title Download images sequentially (Safe from Disconnection) { display-mode: "form" }

import os
import zipfile
from google.colab import files
import time

download_format = "single files" #@param ["zip", "single files"]
CODEFORMER_FIDELITY = 0.7
folder_path = f"/content/CodeFormer/results/user_upload_{CODEFORMER_FIDELITY}/final_results"
zip_name = f"/content/CodeFormer_results_v{CODEFORMER_FIDELITY}.zip"

if os.path.exists(folder_path):
    # Sort files alphabetically to maintain sequential order
    all_files = sorted(os.listdir(folder_path))
    image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp")
    sorted_images = [f for f in all_files if f.lower().endswith(image_extensions)]

    if download_format == "zip":
        print("Zipping files in sequential order...")
        with zipfile.ZipFile(zip_name, 'w') as zipf:
            for file in sorted_images:
                file_path = os.path.join(folder_path, file)
                zipf.write(file_path, arcname=file)
        print("Downloading zip file...")
        files.download(zip_name)
    else:
        print(f"Downloading {len(sorted_images)} files one-by-one instantly...")
        print("This is safe! If connection drops, already downloaded files are saved on your PC.\n")
        for i, file in enumerate(sorted_images, 1):
            file_path = os.path.join(folder_path, file)
            print(f"[{i}/{len(sorted_images)}] Downloading: {file}")
            files.download(file_path)
            # Small delay to prevent browser download queue blockage
            time.sleep(0.5)
else:
    print("Results folder not found. Please make sure the processing cell finished successfully.")

Results folder not found. Please make sure the processing cell finished successfully.
